In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE      = "abfss://bronze@pharmadatalakeuk.dfs.core.windows.net/cdc_demo"
SILVER      = "abfss://silver@pharmadatalakeuk.dfs.core.windows.net/cdc_demo"
SILVER_DELTA = f"{SILVER}/delta"
CHECKPOINT  = f"{SILVER}/_checkpoint"
SCHEMA_LOC  = f"{SILVER}/_schema"
EXPORT      = "abfss://silver@pharmadatalakeuk.dfs.core.windows.net/export/cdc_demo"

In [0]:
def upsert_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    # If the same movement_id appears twice in new files, keep only the newest version
    w = Window.partitionBy("movement_id").orderBy(F.col("last_modified").desc())
    latest = (batch_df
        .withColumn("rn", F.row_number().over(w))
        .filter("rn = 1")
        .drop("rn", "_rescued_data")
        .withColumn("silver_loaded_at", F.current_timestamp()))

    # MERGE into silver: update existing rows, insert new ones
    if DeltaTable.isDeltaTable(spark, SILVER_DELTA):
        (DeltaTable.forPath(spark, SILVER_DELTA).alias("t")
            .merge(latest.alias("s"), "t.movement_id = s.movement_id")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
    else:
        latest.write.format("delta").save(SILVER_DELTA)

    # Export only the changed rows for Snowflake (one new folder per batch)
    latest.write.mode("overwrite").parquet(f"{EXPORT}/batch_{batch_id}")

In [0]:
query = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", SCHEMA_LOC)
    .load(BRONZE)
    .writeStream
    .foreachBatch(upsert_batch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start())

query.awaitTermination()

In [0]:
display(spark.read.format("delta").load(SILVER_DELTA).orderBy("movement_id"))
print("Rows exported for Snowflake:", spark.read.parquet(f"{EXPORT}/*").count())